# 인용 참고문헌 arXiv 메타데이터: 배치 API 수집

참조 논문당 한 행을 만들고 `cit_arxiv_id`에 인용 원본 논문 ID 목록을 저장합니다. `id_list` 배치(100개), 단일 연결, 요청 간 최소 3초를 사용합니다.

In [1]:
import hashlib
import json
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'data' / 'citations_ai').is_dir()), None)
if ROOT is None: raise FileNotFoundError('프로젝트 루트 또는 notebooks/에서 실행하세요.')
INPUT_FILES = [ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part1.jsonl', ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part2.jsonl']
if missing := [p for p in INPUT_FILES if not p.exists()]: raise FileNotFoundError(missing)
OUTPUT_DIR = ROOT / 'data' / 'ai_references' / 'arxiv_ai_references_batched_api'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX, BATCH_SIZE, CHUNK_SIZE = 'arxiv_ai_references_batched_api', 100, 5_000
CACHE_PATH, STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_metadata_cache.jsonl', OUTPUT_DIR / f'{FILE_PREFIX}_state.json'
print(f'입력: {len(INPUT_FILES)}개 / 출력: {OUTPUT_DIR} / 배치: {BATCH_SIZE}')

입력: 2개 / 출력: c:\Users\Playdata\Desktop\arxiv_graph_RAG\data\ai_references\arxiv_ai_references_batched_api / 배치: 100


In [2]:
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path
ARXIV_ID_RE = re.compile(r'^(?:arXiv:)?(?P<id>(?:\d{4}\.\d{4,5}|[A-Za-z-]+(?:\.[A-Za-z-]+)?/\d{7}))(?:v\d+)?$')
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom', 'arxiv': 'http://arxiv.org/schemas/atom'}
OUTPUT_FIELDS = ('id','title','abstract','authors','categories','primary_category','published','updated','doi','pdf_url','source')
def normalize_arxiv_id(value):
    if not isinstance(value, str): return None
    value = value.strip().removeprefix('https://arxiv.org/abs/').removeprefix('http://arxiv.org/abs/')
    match = ARXIV_ID_RE.fullmatch(value)
    return match.group('id') if match else None
def batched(items, size):
    for start in range(0, len(items), size): yield items[start:start + size]
def collect_references(files):
    collected, by_id, skipped = [], {}, 0
    for path in files:
        with Path(path).open(encoding='utf-8-sig') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip(): continue
                try: row = json.loads(line)
                except json.JSONDecodeError as error: raise ValueError(f'{path.name}:{line_number} JSON 파싱 실패') from error
                cit_id = normalize_arxiv_id(row.get('arxiv_id') if isinstance(row, dict) else None)
                if not cit_id: raise ValueError(f'{path.name}:{line_number} 원본 arxiv_id 오류')
                items = row.get('references') or []
                if not isinstance(items, list): raise ValueError(f'{path.name}:{line_number} references는 목록이어야 합니다.')
                for item in items:
                    arxiv_id = normalize_arxiv_id(item.get('arxiv_id') if isinstance(item, dict) else None)
                    if not arxiv_id: skipped += 1; continue
                    record = by_id.get(arxiv_id)
                    if record is None:
                        record = {'arxiv_id': arxiv_id, 'cit_arxiv_id': []}; by_id[arxiv_id] = record; collected.append(record)
                    if cit_id not in record['cit_arxiv_id']: record['cit_arxiv_id'].append(cit_id)
    return collected, skipped
def _text(entry, path):
    node = entry.find(path, ATOM_NS); return node.text.strip() if node is not None and node.text else None
def parse_atom_entries(payload):
    root, result = ET.fromstring(payload), {}
    for entry in root.findall('atom:entry', ATOM_NS):
        arxiv_id = normalize_arxiv_id(_text(entry, 'atom:id'))
        if not arxiv_id: continue
        categories = [node.attrib['term'] for node in entry.findall('atom:category', ATOM_NS) if node.get('term')]
        result[arxiv_id] = {'id': f'https://arxiv.org/abs/{arxiv_id}', 'title': ' '.join((_text(entry, 'atom:title') or '').split()), 'abstract': ' '.join((_text(entry, 'atom:summary') or '').split()), 'authors': [_text(a, 'atom:name') for a in entry.findall('atom:author', ATOM_NS) if _text(a, 'atom:name')], 'categories': categories, 'primary_category': categories[0] if categories else None, 'published': _text(entry, 'atom:published'), 'updated': _text(entry, 'atom:updated'), 'doi': _text(entry, 'arxiv:doi'), 'pdf_url': f'https://arxiv.org/pdf/{arxiv_id}', 'source': 'arxiv'}
    return result
def make_output_record(reference, metadata, found):
    record = {'arxiv_id': reference['arxiv_id'], 'cit_arxiv_id': reference['cit_arxiv_id'], 'found': found}
    record.update({field: (metadata or {}).get(field) for field in OUTPUT_FIELDS} if metadata else {'id':None,'title':None,'abstract':None,'authors':[],'categories':[],'primary_category':None,'published':None,'updated':None,'doi':None,'pdf_url':None,'source':'arxiv'})
    return record

In [3]:
import ssl, time, urllib.error, urllib.parse, urllib.request
import truststore
API_URL, REQUEST_INTERVAL_SECONDS, MAX_RETRIES = 'https://export.arxiv.org/api/query', 3.0, 6
_last_request_at = 0.0
def fetch_batch(arxiv_ids):
    global _last_request_at
    params = {'id_list': ','.join(arxiv_ids), 'max_results': len(arxiv_ids)}
    request = urllib.request.Request(f'{API_URL}?{urllib.parse.urlencode(params)}', headers={'User-Agent':'arxiv-citation-reference-batch-harvester/1.0'})
    for attempt in range(MAX_RETRIES + 1):
        wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - _last_request_at)
        if wait > 0: time.sleep(wait)
        try:
            _last_request_at = time.monotonic()
            with urllib.request.urlopen(request, timeout=120, context=truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)) as response: return parse_atom_entries(response.read())
        except (urllib.error.URLError, TimeoutError) as error:
            if attempt >= MAX_RETRIES: raise
            delay = min(10 * 2 ** attempt, 300); print(f'오류 {error!r}; {delay}초 후 재시도'); time.sleep(delay)
    raise RuntimeError('재시도 횟수 초과')

In [4]:
import os, tempfile
from datetime import datetime, timezone
def atomic_write_jsonl(path, rows):
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, delete=False) as handle:
        temp = Path(handle.name)
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    os.replace(temp, path)
def load_cache():
    if not CACHE_PATH.exists(): return {}
    with CACHE_PATH.open(encoding='utf-8') as handle: return {row['arxiv_id']: {'found':row['found'], 'metadata':row['metadata']} for row in map(json.loads, filter(str.strip, handle))}
references, skipped = collect_references(INPUT_FILES)
target_ids = [r['arxiv_id'] for r in references]
cache, todo = load_cache(), None
todo = [i for i in target_ids if i not in cache]
print(f'고유 참조: {len(references):,}; 캐시됨: {len(cache):,}; 이번 조회: {len(todo):,}; ID 없음: {skipped:,}')
for number, ids in enumerate(batched(todo, BATCH_SIZE), 1):
    found = fetch_batch(ids)
    for arxiv_id in ids: cache[arxiv_id] = {'found': arxiv_id in found, 'metadata': found.get(arxiv_id)}
    atomic_write_jsonl(CACHE_PATH, [{'arxiv_id':k, **v} for k,v in sorted(cache.items())])
    STATE_PATH.write_text(json.dumps({'cached_count':len(cache), 'updated_at':datetime.now(timezone.utc).isoformat()}, ensure_ascii=False), encoding='utf-8')
    print(f'배치 {number}: {len(cache):,}/{len(target_ids):,} 캐시 완료')
records = [make_output_record(r, cache[r['arxiv_id']]['metadata'], cache[r['arxiv_id']]['found']) for r in references]
for index, rows in enumerate(batched(records, CHUNK_SIZE), 1): atomic_write_jsonl(OUTPUT_DIR / f'{FILE_PREFIX}_part{index}.jsonl', rows)
print(f'저장 완료: {len(records):,}행')

고유 참조: 31,913; 캐시됨: 0; 이번 조회: 31,913; ID 없음: 60,549
오류 <HTTPError 429: 'Unknown Error'>; 10초 후 재시도
오류 <HTTPError 429: 'Unknown Error'>; 20초 후 재시도


KeyboardInterrupt: 

In [ ]:
saved = []
for path in sorted(OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl')):
    with path.open(encoding='utf-8') as handle: saved.extend(json.loads(line) for line in handle if line.strip())
assert len(saved) == len(references) == len({row['arxiv_id'] for row in saved})
assert {row['arxiv_id']:row['cit_arxiv_id'] for row in saved} == {row['arxiv_id']:row['cit_arxiv_id'] for row in references}
assert all(isinstance(row['cit_arxiv_id'], list) and row['cit_arxiv_id'] for row in saved)
print(f'검증 완료: 참조 논문 {len(saved):,}행, 인용 관계 {sum(len(row["cit_arxiv_id"]) for row in saved):,}건')
display(saved[:3])